# 03 — Preprocesamiento (Preprocessing)

Pipeline de preprocesamiento del dataset limpio (`wines_SPA_clean.csv`) para preparar los datos antes de introducirlos al modelo de Machine Learning (Recomendador).

**Entrada:** `data/processed/wines_SPA_clean.csv`
**Salida:** `data/processed/wines_SPA_model_ready.csv` y diccionarios de mapeo (`mappings.py`).

**Contexto para Estudiantes de Data Analysis/ML:** Como futuros analistas, sabemos que los modelos matemáticos no entienden texto puro. Necesitamos convertir las categorías en números (Codificación o *Encoding*) y asegurar que todas las variables tengan un "peso" comparable (Escalado o *Scaling*). En este notebook explicaremos paso a paso el porqué de cada transformación, conectando lo que hemos aprendido en el EDA.

In [40]:
# Importamos las librerías fundamentales de Python para el análisis de datos
import pandas as pd # Para la manipulación de tablas (DataFrames)
import numpy as np  # Para operaciones matemáticas avanzadas

# StandardScaler estandariza las características eliminando la media y escalando a varianza unitaria.
# Es fundamental para algoritmos basados en distancias (como KNN para recomendaciones).
from sklearn.preprocessing import StandardScaler

# Definimos las rutas constantes de entrada y salida (buena práctica de código profesional)
INPUT_PATH = '../data/processed/wines_SPA_clean.csv'
OUTPUT_CSV_PATH = '../data/processed/wines_SPA_model_ready.csv'
OUTPUT_DICT_PATH ='../preprocessing/mappings.py'

## 1. Carga de datos y Validación Residual

Antes de transformar, cargamos el CSV y aplicamos un control de calidad final. En Machine Learning, un modelo entrenado con "basura" dará predicciones "basura" (*Garbage In, Garbage Out*). Buscamos confirmar que:
1. No queden valores nulos residuales.
2. Los tipos de datos (dtypes) sean los correctos.
3. No existan inconsistencias categóricas.

In [41]:
# 1. Cargamos el dataset limpio usando pandas
df = pd.read_csv(INPUT_PATH)
print("Forma inicial del dataset:", df.shape)

# 2. Validación de nulos
# El método .isna().sum() nos da el conteo exacto de celdas vacías por cada columna.
null_counts = df.isna().sum()
print("\n--- Nulos Residuales ---")
print(null_counts[null_counts > 0]) # Solo mostramos si hay alguna con nulos

# 3. Validación de tipos de datos
# Comprobamos que las variables matemáticas sean int/float y las textuales sean object.
print("\n--- Tipos de Datos ---")
print(df.dtypes)

# 4. Chequeo de consistencia de negocio (Validación extra)
# Corroboramos que un vino Tinto no tenga asociada una variedad de uva blanca exclusivamente.
red_varieties = {'Tempranillo', 'Monastrell', 'Grenache', 'Mencia', 'Bobal', 'Syrah', 'Cabernet Sauvignon'}
white_varieties = {'Albarino', 'Verdejo', 'Chardonnay', 'Godello', 'Treixadura', 'Palomino', 'Viura', 'Sauvignon Blanc'}

inconsistencias = df[
    ((df['vine_type'] == 'Blanco') & (df['grape_variety'].isin(red_varieties))) |
    ((df['vine_type'] == 'Tinto') & (df['grape_variety'].isin(white_varieties)))
]
print(f"\nFilas inconsistentes detectadas: {len(inconsistencias)}")

Forma inicial del dataset: (2024, 13)

--- Nulos Residuales ---
flavor_descriptor    106
dtype: int64

--- Tipos de Datos ---
winery                  object
wine_name               object
year                     int64
rating                 float64
region                  object
price_euros            float64
vine_type               object
grape_variety           object
wine_ageing              int64
service_temperature     object
quality_price_ratio    float64
luxury_category          int64
flavor_descriptor       object
dtype: object

Filas inconsistentes detectadas: 0


## 2. Escalado de variables numéricas: El Precio

Como detectamos en la fase exploratoria (EDA), el precio (`price_euros`) tiene una asimetría muy fuerte (una "cola larga" a la derecha, con valores extremos de hasta miles de euros). Esto distorsionaría cualquier cálculo de distancia porque las diferencias absolutas en los vinos de lujo opacarían al resto.

Aplicar un logaritmo (`log1p`) comprime esta asimetría, acercando los valores a una distribución normal y haciendo que el modelo evalúe mejor los saltos proporcionales.

In [42]:
# Transformación logarítmica del precio
# Usamos np.log1p(x) que calcula log(1 + x). Es más robusto que np.log() porque evita errores si algún valor fuera 0.
df['price_log'] = np.log1p(df['price_euros'])

# Ojo: Mantenemos la columna original 'price_euros' intacta porque la vamos a usar a continuación 
# como nuestra "variable objetivo" para codificar las regiones.

## 3. Codificación de variables categóricas (Encoding)

Los algoritmos de ML solo pueden digerir números. Aquí es vital justificar la técnica que elegimos según el número de categorías únicas (cardinalidad) que tenga nuestra variable:

*   **`vine_type` (5 tipos): Usaremos One-Hot Encoding**.
    Al tener muy pocas categorías, creamos una columna binaria (1 o 0) por cada tipo (ej. `type_Tinto`, `type_Blanco`). Como son pocas, no saturamos el modelo con demasiadas dimensiones.

*   **`region` (76) y `grape_variety` (16): Usaremos Target Encoding (Precio Medio)**.
    Si aplicáramos One-Hot a las 76 regiones, añadiríamos 76 columnas nuevas, la mayoría llenas de ceros, lo que causa la *maldición de la dimensionalidad*. En su lugar, calculamos el **precio medio de cada región/uva** y reemplazamos el texto de la región por ese valor continuo. En un motor de recomendación, el precio promedio de una D.O. es un indicador semántico valioso para agrupar vinos similares (ej. regiones premium vs regiones económicas).

In [43]:
# --- A) One-Hot Encoding para el color del vino ---
# pd.get_dummies crea estas columnas bandera.
# En recomendadores (ML No Supervisado) solemos dejar todas las columnas (drop_first=False) 
# para que la distancia matemática trate todos los tipos por igual.
df_encoded = pd.get_dummies(df, columns=['vine_type'], prefix='type', dtype=int)

# --- B) Target Encoding (basado en precio medio) para la zona y la uva ---
# 1. Creamos diccionarios calculando el precio medio (.mean()) agrupado por región/uva.
region_price_map = df.groupby('region')['price_euros'].mean().round(2).to_dict()
grape_price_map = df.groupby('grape_variety')['price_euros'].mean().round(2).to_dict()

# 2. Reemplazamos la categoría de texto por el valor numérico correspondiente usando .map()
# Lo guardamos en columnas nuevas para no perder los textos originales, que nos sirven para mostrar en la web/app.
df_encoded['region_encoded'] = df['region'].map(region_price_map)
df_encoded['grape_variety_encoded'] = df['grape_variety'].map(grape_price_map)

print("Columnas One-Hot generadas para el tipo de vino:")
print([c for c in df_encoded.columns if 'type_' in c])

Columnas One-Hot generadas para el tipo de vino:
['type_Blanco', 'type_Desconocido', 'type_Espumoso', 'type_Generoso', 'type_Tinto']


## 4. Estandarización Final con StandardScaler

Nuestras variables numéricas deben hablar "el mismo idioma". Imagina combinar el Año (valores ~2015) con el Rating (valores ~4.5). Sin ajustar, el Año dominaría por completo cualquier cálculo matemático.

`StandardScaler` toma cada columna numérica y la reescala para que su promedio sea 0 y su desviación estándar sea 1. Así, todas las variables contribuyen equitativamente en la matriz de distancias del modelo final.

In [44]:
# Seleccionamos explícitamente las columnas (features) que se usarán en el modelo.
# Descartamos las variables de texto y variables redundantes (ej. price_euros, ya que usamos price_log).
# Nota: La columna quality_price_ratio ya correlaciona negativamente con el lujo y el precio (como vimos en el EDA),
# lo cual es normal y ayudará al recomendador a perfilar los segmentos de precio/calidad.
numerical_features = [
    'year', 'rating', 'price_log',
    'wine_ageing', 'quality_price_ratio', 'luxury_category',
    'region_encoded', 'grape_variety_encoded'
]

# Añadimos las columnas generadas por el One-Hot Encoding
type_cols = [c for c in df_encoded.columns if c.startswith('type_')]
model_features = numerical_features + type_cols

# Instanciamos el estandarizador
scaler = StandardScaler()

# Ajustamos (fit) el escalador a nuestros datos y los transformamos (transform) de un golpe.
# Asignamos el resultado a nuevas columnas con el sufijo '_scaled' por limpieza.
scaled_cols = [f"{col}_scaled" for col in model_features]
df_encoded[scaled_cols] = scaler.fit_transform(df_encoded[model_features])

print("Primeras filas con las variables estandarizadas, listas para la matriz de similitud:")
display(df_encoded[scaled_cols].head(3))

Primeras filas con las variables estandarizadas, listas para la matriz de similitud:


,year_scaled,rating_scaled,price_log_scaled,wine_ageing_scaled,quality_price_ratio_scaled,luxury_category_scaled,region_encoded_scaled,grape_variety_encoded_scaled,type_Blanco_scaled,type_Desconocido_scaled,type_Espumoso_scaled,type_Generoso_scaled,type_Tinto_scaled
0,0.157751,3.395852,2.617543,-0.509401,-1.124180,0.955533,0.963549,0.495127,-0.269448,-0.182154,-0.138325,-0.215721,0.441367
1,0.609746,3.395852,1.503525,-0.509401,-0.995762,0.955533,0.794731,0.495127,-0.269448,-0.182154,-0.138325,-0.215721,0.441367
2,-0.203844,2.711883,1.538083,-0.509401,-1.007436,0.955533,1.036882,0.495127,-0.269448,-0.182154,-0.138325,-0.215721,0.441367


## 5. Creación de diccionarios de mapeo para Producción (.py)

Para que la aplicación final interactúe con el usuario, necesitamos generar diccionarios que conecten la lógica del recomendador con la interfaz. Estos diccionarios los guardaremos en un archivo Python (`mappings.py`) que será importado directamente por nuestro backend.

**Enfoque Profesional (Producción):** En este bloque usamos la estructura optimizada. Al crear un diccionario donde el "sabor" es la llave (Key) y la "familia" es el valor (Value), logramos que las búsquedas de Python sean de complejidad $\mathcal{O}(1)$. Esto hace que la aplicación final procese datos mucho más rápido al no tener que recorrer listas internas.

In [45]:
py_content = '''mappings.py
===========
Diccionarios de mapeo semántico para Sommelier IA.

Contiene dos mapeos:
    1. FLAVOR_SEMANTIC_MAP: cada descriptor de sabor individual -> su categoría
         semántica (familia aromática). Permite pasar de 51 columnas dummy
         (una por descriptor) a solo 5 features agregadas por familia, mucho
         más manejable para el motor de similitud.
    2. SERVING_TEMP_SEMANTIC_MAP: cada rango de temperatura de servicio ->
         una etiqueta semántica ordinal + su punto medio numérico (para poder
         tratarla como variable numérica si el modelo lo necesita).

Basado en el vocabulario real presente en wines_SPA_enriched_FINAL.csv
(51 descriptores únicos tras fusionar sinónimos ES/EN) y en las 5 familias
usadas originalmente en FLAVOR_KEYWORDS del scraper.

# ─────────────────────────────────────────────────────────────────────────────
# 1. FLAVOR_DESCRIPTOR -> FAMILIA AROMÁTICA
# ─────────────────────────────────────────────────────────────────────────────
FLAVOR_SEMANTIC_MAP: dict = {
        # Frutas rojas y negras
        "Cherry": "fruta_roja_negra", "Blackberry": "fruta_roja_negra", "Plum": "fruta_roja_negra",
        "Strawberry": "fruta_roja_negra", "Raspberry": "fruta_roja_negra", "Cassis": "fruta_roja_negra",
        "Blueberry": "fruta_roja_negra", "Fig": "fruta_roja_negra",

        # Frutas blancas y tropicales
        "Peach": "fruta_blanca_tropical", "Apricot": "fruta_blanca_tropical", "Citrus": "fruta_blanca_tropical",
        "Lemon": "fruta_blanca_tropical", "Orange": "fruta_blanca_tropical", "Apple": "fruta_blanca_tropical",
        "Pear": "fruta_blanca_tropical", "Melon": "fruta_blanca_tropical", "Tropical": "fruta_blanca_tropical",
        "Pineapple": "fruta_blanca_tropical", "Mango": "fruta_blanca_tropical",

        # Madera y especias
        "Oak": "madera_especias", "Vanilla": "madera_especias", "Cedar": "madera_especias",
        "Tobacco": "madera_especias", "Leather": "madera_especias", "Smoke": "madera_especias",
        "Toast": "madera_especias",

        # Confitería y tierra
        "Chocolate": "confiteria_tierra", "Coffee": "confiteria_tierra", "Caramel": "confiteria_tierra",
        "Licorice": "confiteria_tierra", "Anise": "confiteria_tierra", "Spice": "confiteria_tierra",
        "Pepper": "confiteria_tierra", "Clove": "confiteria_tierra", "Cinnamon": "confiteria_tierra",
        "Truffle": "confiteria_tierra", "Mushroom": "confiteria_tierra", "Earthy": "confiteria_tierra",

        # Florales y minerales
        "Herb": "floral_mineral", "Grass": "floral_mineral", "Floral": "floral_mineral",
        "Rose": "floral_mineral", "Violet": "floral_mineral", "Mineral": "floral_mineral",
        "Saline": "floral_mineral", "Slate": "floral_mineral", "Almond": "floral_mineral",
        "Hazelnut": "floral_mineral", "Honey": "floral_mineral", "Dried Fruit": "floral_mineral",
        "Raisin": "floral_mineral",
}

FLAVOR_FAMILIES: list = [
        "fruta_roja_negra", "fruta_blanca_tropical", "madera_especias",
        "confiteria_tierra", "floral_mineral",
]


def map_flavors_to_families(flavor_descriptor: str) -> dict:
        """Convierte un string 'Cherry, Oak, Vanilla' en un dict de conteos por familia,
        ej. {'fruta_roja_negra': 1, 'madera_especias': 2, ...}.
        Términos no reconocidos en FLAVOR_SEMANTIC_MAP se ignoran (no deberían
        aparecer si el dato viene de flavor_descriptor ya limpio)."""
        counts = {fam: 0 for fam in FLAVOR_FAMILIES}
        if not isinstance(flavor_descriptor, str):
                return counts
        for token in flavor_descriptor.split(","):
                family = FLAVOR_SEMANTIC_MAP.get(token.strip())
                if family:
                        counts[family] += 1
        return counts


# ─────────────────────────────────────────────────────────────────────────────
# 2. SERVING_TEMPERATURE -> CATEGORÍA SEMÁNTICA + PUNTO MEDIO NUMÉRICO
# ─────────────────────────────────────────────────────────────────────────────
SERVING_TEMP_SEMANTIC_MAP: dict = {
        "6-8°C":   {"label": "muy_frio",        "midpoint_celsius": 7.0},
        "8-10°C":  {"label": "frio",            "midpoint_celsius": 9.0},
        "10-12°C": {"label": "fresco",          "midpoint_celsius": 11.0},
        "10-14°C": {"label": "fresco",          "midpoint_celsius": 12.0},
        "12-14°C": {"label": "fresco_templado", "midpoint_celsius": 13.0},
        "16-18°C": {"label": "templado",        "midpoint_celsius": 17.0},
}


def map_temperature_to_semantic(service_temperature: str) -> tuple:
        """Devuelve (etiqueta_semantica, punto_medio_en_celsius) para un rango
        de temperatura de servicio. Si el rango no está en el mapa, devuelve
        ('desconocido', None)."""
        entry = SERVING_TEMP_SEMANTIC_MAP.get(service_temperature)
        if entry is None:
                return "desconocido", None
        return entry["label"], entry["midpoint_celsius"]
'''

with open(OUTPUT_DICT_PATH, 'w', encoding='utf-8') as f:
    f.write(py_content)

print(f"✅ Archivo '{OUTPUT_DICT_PATH}' exportado correctamente para uso en backend.")


✅ Archivo '../preprocessing/mappings.py' exportado correctamente para uso en backend.


## 6. Output Entregable: Exportación del DataFrame final

Verificamos la creación de la tabla definitiva. Este DataFrame contiene tanto las variables originales como las preparadas para el ML, por lo que es la única fuente de la verdad para el modelo.

In [46]:
# Guardamos el dataset final procesado en formato CSV
df_encoded.to_csv(OUTPUT_CSV_PATH, index=False, encoding='utf-8-sig')

print(f"🚀 Pipeline de preprocesamiento completado con éxito.")
print(f"Dataset de {df_encoded.shape[0]} filas y {df_encoded.shape[1]} columnas guardado en '{OUTPUT_CSV_PATH}'.")

🚀 Pipeline de preprocesamiento completado con éxito.
Dataset de 2024 filas y 33 columnas guardado en '../data/processed/wines_SPA_model_ready.csv'.
